# Valuation of Performance Share Units (PSUs) under IFRS 2: A Technical Perspective

## 1. Introduction and The Vesting Framework
In equity compensation (under IFRS 2 or ASC 718), "vesting" represents the process by which a beneficiary earns the non-forfeitable right to receive shares. This right is contingent upon satisfying specific requirements, which fall into three main categories:

* **Service Conditions:** Require only that the employee continues to provide services to the entity for a specified period (retention).
* **Market Conditions:** Linked to the performance of the company's equity (or related derivatives), such as achieving a Target Price or a specific Relative Total Shareholder Return (TSR) against a peer group.
* **Non-Market Performance Conditions:** Tied to internal financial, operational, or strategic metrics (e.g., EPS, EBITDA targets, or specific project milestones).

### 1.1 The Valuation Asymmetry
IFRS 2 dictates a strict asymmetry in handling these conditions. Market conditions are incorporated directly into the grant-date fair value using quantitative models under the risk-neutral measure ($Q$). Conversely, Non-Market and Service conditions do not affect the initial fair value unit but act stochastically on the estimated number of instruments expected to vest under the real-world measure ($P$), allowing for subsequent true-up adjustments to the income statement.

### 1.2 Payout Multipliers and Path-Dependency
In advanced PSU structures, performance conditions rarely act as simple binary triggers (cliff vesting). Instead, the contract often introduces a **multiplicative component** defined by a payout curve. The payoff is scaled based on the degree of achievement: e.g., 50% vesting at a minimum threshold, 100% at target, and up to a 200% cap for outperformance.

Crucially, these conditions make the structure deeply **path-dependent**. Vesting and multipliers often depend on the entire price evolution over time rather than just the terminal value ($S_T$). Examples include averaging mechanisms (Asian-style features over the last 60-90 days), high-water marks, or laddered barriers. This path dependency renders closed-form solutions (like the Black-Scholes formula) inapplicable, making Monte Carlo simulation the standard numerical approach.

## 2. Technical Framework for PSU Valuation
PSUs are contracts whose payoff depends on one or more stochastic drivers and an indicator of vesting. The grant-date fair value is:
$$FV_{grant} = E_Q \left[ D(0,T) \cdot H(S_{\cdot}, M_{\cdot}) \cdot 1_{\{Vesting\}} \right]$$
where
- $D(0,T) = \exp\left(-\int_0^T r_t \, dt\right)$
- $S_{\cdot}$ denotes tradable market factors
- $M_{\cdot}$ denotes non-market metrics
- $1_{\{Vesting\}} = 1$ if all vesting conditions are satisfied

For equity settlement, payoff can be written as
$$H = S_T \cdot g(S_{\cdot}, M_{\cdot})$$
or, when the award is expressed as a share count,
$$H = N(S_{\cdot}, M_{\cdot}) \cdot S_T$$
with payout functions $g(\cdot)$ and $N(\cdot)$ incorporating caps, floors, linear/curved payout schedules, and partial vesting.

### 2.1 Market versus Non-Market Conditions
Market conditions:
- valued under risk-neutral measure $Q$
- modeled using SDEs such as
$$dS_t = S_t (r - q) \, dt + S_t \sigma \, dW_t^Q$$

Non-market conditions:
- reflected through probability of attainment $p_{\text{attain}}$
- not remeasured as market instruments
- incorporated as an adjustment factor:
$$FV_{grant} = E_Q \left[ D(0,T) \cdot H(S_{\cdot}) \cdot 1_{\{\text{MarketConditions}\}} \right] \cdot p_{\text{nonmarket}}$$

When non-market metrics are stochastic:
$$FV_{grant} = E \left[ D(0,T) \cdot H(S_{\cdot}, M_{\cdot}) \cdot 1_{\{Vesting\}} \right]$$
with $M_{\cdot}$ simulated under a real-world measure $P$ or calibrated to empirical distributions.

### 2.2 Vesting and Service Probabilities
Service conditions are modeled via a survival function:
$$P_{\text{service}}(t) = \exp\left(-\int_0^t \lambda(u) \, du\right)$$

For graded vesting over tranches $\{t_i\}$,
$$FV_{grant} = \sum_{i} E_Q \left[ D(0,t_i) \cdot H_i \cdot 1_{\{Vesting_i\}} \right] \cdot P_{\text{service}}(t_i)$$

For discrete vesting with hurdle thresholds $H_k$ and payout curves $f(x)$,
$$\text{Payoff}_i = f(\text{Performance}_i) \cdot S_{t_i}$$

## 3. Monte Carlo Justification
Monte Carlo is necessary because PSUs often feature:
- path-dependent vesting (averaging, high-water marks, multi-period TSR)
- multidimensional underlyings (peer group returns, correlated KPIs)
- discontinuous payoffs (laddered or barrier-based vesting)
- mixed conditions requiring joint simulation of $S$ and $M$

The numerical estimator is:
$$FV_N = \frac{1}{N} \sum_{j=1}^N D_j \cdot H_j \cdot 1_{\{Vesting\}_j}$$
with standard error $\approx \sigma_H / \sqrt{N}$ and potential variance reduction using antithetic sampling, control variates, or quasi-Monte Carlo.

## 4. Formal Modelling of Conditions
Service condition:
- hazard $\lambda(t)$, survival $S_{\text{srv}}(t) = \exp\left(-\int_0^t \lambda(u) \, du\right)$

Market condition:
- indicator $1_{\{\text{Market}\}} = 1$ if relative TSR or index-based hurdle is met
- example for relative TSR rank:
$$1_{\{Rank \ge \alpha\}} = 1_{\{\mathrm{TSR}_{company} / \mathrm{TSR}_{peer} \ge \text{threshold}\}}$$

Non-market condition:
- attainment probability $p_{\text{attain}} = P_P(M_T \ge \text{target})$
- for multiple conditions:
$$1_{\{Vesting\}} = 1_{\{\text{Market}\}} \cdot 1_{\{\text{NonMarket}\}} \cdot 1_{\{\text{Service}\}}$$
and if non-market metrics are stochastic,
$$E[1_{\{\text{NonMarket}\}}] = \int 1_{\{m \text{ meets target}\}} \, f_M(m) \, dm$$

## 5. Accounting Implications
Grant-date measurement:
- $FV_{grant}$ is recognized over the service period
- expense for period $t$:
$$\text{Expense}_t = FV_{grant} \times \Delta P_{\text{service}}(t)$$

For a total grant of $N$ awards:
$$\text{Total\_cost} = N \times FV_{grant}$$

If vesting is graded, allocate expense by tranche weights and expected survival.

Changes in estimates:
- service forfeiture updates affect future expense
- market conditions remain priced in at grant date
- non-market conditions are reflected only through probability updates when required by IFRS 2 guidance

## 6. Implementation Principles
- simulate $S_t$ under risk-neutral dynamics calibrated to implied vol and dividend yield
- simulate $M_t$ under real-world models or scenario distributions
- apply correlation structure between market and non-market drivers if dependencies exist
- use variance reduction and ensure enough paths for target standard error
- verify with sensitivity analysis on $\sigma$, $r$, $q$, correlation, $p_{\text{attain}}$, and vesting hazards

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
from dateutil.relativedelta import relativedelta

import numpy as np

def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths, q=0.0, corr_matrix=None, seed=42):
    np.random.seed(seed)

    S0_vec = np.atleast_1d(S0)
    sigma_vec = np.atleast_1d(sigma)
    q_vec = np.atleast_1d(q)
    
    n_assets = len(S0_vec)
    
    if len(q_vec) == 1 and n_assets > 1:
        q_vec = np.repeat(q_vec, n_assets)
        
    if corr_matrix is None:
        corr_matrix = np.eye(n_assets)
    else:
        corr_matrix = np.atleast_2d(corr_matrix)
        
    dt = T / n_steps
    
    L = np.linalg.cholesky(corr_matrix)
    Z = np.random.standard_normal((n_assets, n_paths, n_steps))
    W = np.tensordot(L, Z, axes=[1, 0])
    
    S = np.zeros((n_assets, n_paths, n_steps + 1))
    for i in range(n_assets):
        S[i, :, 0] = S0_vec[i]
        
    for t in range(n_steps):
        for i in range(n_assets):
            drift = (r - q_vec[i] - 0.5 * sigma_vec[i]**2) * dt
            diffusion = sigma_vec[i] * np.sqrt(dt) * W[i, :, t]
            S[i, :, t + 1] = S[i, :, t] * np.exp(drift + diffusion)
            
    if n_assets == 1:
        return S[0] # Rimuove la 3a dimensione, restituisce (n_paths, n_steps + 1)
    else:
        return S    # Restituisce il tensore 3D completo: (n_assets, n_paths, n_steps + 1)

# Mrkt data
stock = "NVDA"
index = "^GSPC"
portfolio = {
    "NVDA": 150000,
    "AAPL": 50000, 
    "MSFT": 20000    
}
cash_and_equivalents = 5000000
unlisted_assets = 10000000
pfn = -5000000

r = 0.03
q = 0.02

# Simulation data
Date_Start = "2024-07-01"
Date_End = "2026-12-31"
Valdate = "2024-01-01"

n_paths = 100000
av_t_S = 30
av_t_I = 1
av_t_NAV = 1
day_count = 260

start_dt = pd.to_datetime(Date_Start)
end_dt   = pd.to_datetime(Date_End)
val_dt   = pd.to_datetime(Valdate)
T = (end_dt - start_dt).days / 365.0
n_steps = day_count * T

## Index data
data = yf.download(index, start=start_dt+relativedelta(years=-1), end=start_dt, progress=False)
prezzi_I = data["Close"].squeeze() 
returns_I = 100 * np.log(prezzi_I / prezzi_I.shift(1))
returns_I = returns_I.dropna()
returns_I.name = "returns_Index"
I0 = prezzi_I.tail(av_t_I).mean()
sigma_I = returns_I.std() * np.sqrt(day_count)
## Stock data
data = yf.download(stock, start=start_dt+relativedelta(years=-1), end=start_dt, progress=False)
prezzi_S = data["Close"].squeeze()
returns_S = 100 * np.log(prezzi_S / prezzi_S.shift(1))
returns_S = returns_S.dropna()
returns_S.name = "returns_Stock"
S0 = prezzi_S.tail(av_t_S).mean()
sigma_S = returns_S.std() * np.sqrt(day_count)
## NAV data
tickers = list(portfolio.keys())
data = yf.download(tickers, start=start_dt+relativedelta(years=-1), end=start_dt, progress=False)["Close"]
for ticker, shares in portfolio.items():
    data[ticker] = data[ticker] * shares
data["Listed_GAV"] = data.sum(axis=1)
data["Total_NAV"] = data["Listed_GAV"] + cash_and_equivalents + unlisted_assets + pfn
prezzi_NAV = data["Total_NAV"]
returns_NAV = 100 * np.log(prezzi_NAV / prezzi_NAV.shift(1))
returns_NAV = returns_NAV.dropna()
returns_NAV.name = "returns_NAV"
nav0 = prezzi_NAV.tail(av_t_NAV).mean()
sigma_NAV = returns_NAV.std() * np.sqrt(day_count)
## Correlation matrix
returns_df = pd.DataFrame({
    "Stock": returns_S,
    "Index": returns_I,
    "NAV": returns_NAV
})
corr_matrix = returns_df.dropna().corr()

## discount factor
disc = np.exp(-r * T)

# Configurazione del piano LTI
lti_config = {
    "plan_details": {
        "grant_date": Date_Start,
        "maturity_date": Date_End,
    },
    "conditions": {
        "TSR": {
            "type": "absolute", 
            "weight": 0.50,   
            "start_value": S0, 
            "x_nodes": [-np.inf, 0.033, 0.063, 0.158, np.inf],
            "y_nodes": {
                "CEO":   [0.0, 0.5, 1.0, 3.0, 3.0],
                "OTHER": [0.0, 0.5, 1.0, 2.0, 2.0]
            }
        },
        "NAV": {
            "type": "relative_to_index",
            "weight": 0.50,
            "start_asset": nav0,
            "start_index": I0,
            "x_nodes": [-np.inf, 0.0, 0.063, np.inf],
            "y_nodes": {
                "CEO":   [0.0, 1.0, 3.0, 3.0],
                "OTHER": [0.0, 1.0, 2.0, 2.0]
            }
        }
    }
}

# --- 1. PREPARAZIONE DEGLI INPUT VETTORIALI ---
n_steps = int(day_count * T)
S0_vec = np.array([S0, I0, nav0])
sigma_vec = np.array([sigma_S / 100.0, sigma_I / 100.0, sigma_NAV / 100.0])
q_vec = np.array([q, 0.0, q]) 
corr_array = corr_matrix.values

# --- 2. GENERAZIONE DELLE TRAIETTORIE ---
print("Generazione delle traiettorie Monte Carlo in corso...")

trajectories = simulate_gbm_paths(
    S0=S0_vec, 
    r=r, 
    sigma=sigma_vec, 
    T=T, 
    n_steps=n_steps, 
    n_paths=n_paths, 
    q=q_vec, 
    corr_matrix=corr_array
)
S_paths   = trajectories[0]
I_paths   = trajectories[1]
nav_paths = trajectories[2]

# --- 3. CALCOLO DELLE CONDIZIONI DI VESTING ---
S_end = S_paths[:, -av_t_S:].mean(axis=1)
S_spot = S_paths[:, -1] # Prezzo spot a scadenza per il calcolo del valore monetario
I_end = I_paths[:, -av_t_I:].mean(axis=1)  # Media degli ultimi giorni per ridurre il noise
nav_end = nav_paths[:, -av_t_NAV:].mean(axis=1)  # Media degli ultimi giorni per ridurre il noise
cagr_power = 1/3

tsr_cagr = (S_end / lti_config["conditions"]["TSR"]["start_value"]) ** cagr_power - 1
nav_cagr = (nav_end / lti_config["conditions"]["NAV"]["start_asset"]) ** cagr_power - 1
msci_cagr = (I_end / lti_config["conditions"]["NAV"]["start_index"]) ** cagr_power - 1


relative_performance = nav_cagr - msci_cagr

# --- 4. ORCHESTRATORE GENERALIZZATO PER TUTTI I RUOLI ---
def apply_curve(performance, config_node, role):
    return np.interp(performance, config_node["x_nodes"], config_node["y_nodes"][role])

# Estraiamo dinamicamente tutti i ruoli configurati nel dizionario (es. "CEO", "OTHER")
ruoli_disponibili = list(lti_config["conditions"]["TSR"]["y_nodes"].keys())

risultati = {}

for role in ruoli_disponibili:
    # Applicazione curve per lo specifico ruolo
    mult_tsr = apply_curve(tsr_cagr, lti_config["conditions"]["TSR"], role)
    mult_nav = apply_curve(relative_performance, lti_config["conditions"]["NAV"], role)
    
    # Moltiplicatore Totale (Media pesata)
    total_perf_multiplier = (mult_tsr * lti_config["conditions"]["TSR"]["weight"]) + \
                            (mult_nav * lti_config["conditions"]["NAV"]["weight"])
    
    # Calcolo Payoff e Sconto
    payoffs = total_perf_multiplier * S_spot
    discounted_payoffs = payoffs * disc
    
    # Metriche
    fv_mc = np.mean(discounted_payoffs)
    se_mc = np.std(discounted_payoffs) / np.sqrt(n_paths)
    
    # Salvataggio nel dizionario dei risultati
    risultati[role] = {
        "Fair Value (Market) €": round(fv_mc, 4),
        "Standard Error": round(se_mc * np.sqrt(day_count), 4),
        "Prob. Vesting": np.mean(total_perf_multiplier > 0),
    }

# --- 5. OUTPUT DEI RISULTATI IN FORMATO TABELLARE ---
df_risultati = pd.DataFrame(risultati).T
prob_vesting = np.mean(total_perf_multiplier > 0)

print("\n" + "=" * 80)
print("RISULTATI VALUTAZIONE LTI")
print("=" * 80)
print(df_risultati.to_string())
print("-" * 80)

Generazione delle traiettorie Monte Carlo in corso...

RISULTATI VALUTAZIONE LTI
        Fair Value (Market) €  Standard Error  Prob. Vesting
CEO                  194.6136         14.8391            1.0
OTHER                146.4294          9.7906            1.0
OTHER2               128.9294          7.2906            1.0
--------------------------------------------------------------------------------
